In [1]:
import pymssql
import pandas as pd
DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

In [2]:
import datetime as dt
today = dt.date.today()
today

datetime.date(2026, 9, 29)

In [ ]:
with pymssql.connect(DB_info['server'], DB_info['user'], DB_info['password']) as conn:
    cursor = conn.cursor()

    sql = '''
    DECLARE @report_date date = CAST(GETDATE() AS date);

    WITH paid_orders AS (           
        SELECT
            tao.ORPoiId,
            tao.PoiId,
            CAST(tao.PaymentTime AS date) AS payment_date,
            tao.Status
        FROM mars.dbo.TakeAwayOrder tao WITH (NOLOCK)
        WHERE tao.PaymentTime >= DATEADD(day, -30, @report_date)
          AND tao.PaymentTime < @report_date
          AND tao.PaymentTime IS NOT NULL
          AND tao.Status IN (0, 4, 10)           --"AutoDeclined": "0",  "CancelledByUser": "4", "Completed": "10"
    ),
    order_counts AS (
        SELECT
            ORPoiId,        -- tao.ORPoiId
            PoiId,          -- tao.PoiId
            SUM(CASE WHEN payment_date = DATEADD(day, -1, @report_date) AND Status IN (0, 4) THEN 1 ELSE 0 END) AS day_1_cancel_order,
            SUM(CASE WHEN payment_date = DATEADD(day, -1, @report_date) AND Status = 10 THEN 1 ELSE 0 END) AS day_1_completed_order,
            SUM(CASE WHEN payment_date = DATEADD(day, -2, @report_date) AND Status IN (0, 4) THEN 1 ELSE 0 END) AS day_2_cancel_order,
            SUM(CASE WHEN payment_date = DATEADD(day, -2, @report_date) AND Status = 10 THEN 1 ELSE 0 END) AS day_2_completed_order,
            SUM(CASE WHEN payment_date = DATEADD(day, -3, @report_date) AND Status IN (0, 4) THEN 1 ELSE 0 END) AS day_3_cancel_order,
            SUM(CASE WHEN payment_date = DATEADD(day, -3, @report_date) AND Status = 10 THEN 1 ELSE 0 END) AS day_3_completed_order,
            SUM(CASE WHEN Status IN (0, 4) THEN 1 ELSE 0 END) AS past_30_days_cancel_order,
            SUM(CASE WHEN Status = 10 THEN 1 ELSE 0 END) AS past_30_days_completed_order
        FROM paid_orders
        GROUP BY ORPoiId, PoiId
    )
    SELECT                                   -- takeawayorder join mars.poi
        p.ORPoiId AS [OR POIID],
        p.PoiId AS [BBO POIID],               -- mars poiid
        p.NameLang1 AS [Restaurant Name(Lang 1)],
        CONCAT(oc.day_1_cancel_order, '/', oc.day_1_completed_order) AS [Day-1 (Canceled by System & Customer/Completed Order)],
        CONCAT(oc.day_2_cancel_order, '/', oc.day_2_completed_order) AS [Day-2 (Canceled by System & Customer/Completed Order)],
        CONCAT(oc.day_3_cancel_order, '/', oc.day_3_completed_order) AS [Day-3 (Canceled by System & Customer/Completed Order)],
        CONCAT(oc.past_30_days_cancel_order, '/', oc.past_30_days_completed_order) AS [Past 30 Days (Canceled by System & Customer/Completed Order)]
    FROM order_counts oc
    INNER JOIN mars.dbo.Poi p WITH (NOLOCK)
        ON (p.ORPoiId = oc.ORPoiId and p.PoiId = oc.PoiId)
    WHERE p.Status in (10, 3)         -- 10: normal, 3: renovate
      AND oc.day_1_cancel_order > 0            -- Only  include Day-1 cancelled order (by system & user) > 0 POI
    ORDER BY p.ORPoiId, p.PoiId;
    '''

    df = pd.read_sql(sql, conn)
df

C:\Users\lenalee\AppData\Local\Temp\ipykernel_58088\893308902.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,OR POIID,BBO POIID,Restaurant Name(Lang 1),Day-1 (Canceled by System & Customer/Completed Order),Day-2 (Canceled by System & Customer/Completed Order),Day-3 (Canceled by System & Customer/Completed Order),Past 30 Days (Canceled by System & Customer/Completed Order)
0,30675,3866,大雞大超級雞排,1/7,0/2,0/1,2/92
1,174287,16411,Greyhound Café,1/0,0/1,0/3,4/25
2,483856,18932,祥興記上海生煎包,1/0,0/2,0/2,1/25
3,536564,4159,霸嗎拉麵,1/1,0/0,0/1,2/9
4,661814,23381,花園餐廳,1/0,0/0,0/0,1/0
5,683970,26455,鮨一日本料理,1/0,0/0,0/0,1/1
6,697162,26575,手作功夫茶,1/0,0/1,0/3,7/21
7,705578,27708,Mr. OJ,1/4,0/0,0/0,8/36
8,729999,141733,大糖,1/1,0/0,0/0,2/2
9,730013,29678,營豐海南雞專門店,1/1,0/0,0/2,4/13


In [8]:
filename = 'TA_Cancel_Order_Daily_Report-%s.xlsx'%str(today)
df.to_excel(filename, index=False)
filename

'TA_Cancel_Order_Daily_Report-2026-09-29.xlsx'